# THUẬT TOÁN PHÂN CỤM DỰA TRÊN LƯỚI KẾT HỢP KỸ THUẬT PHÁT HIỆN RANH GIỚI

---

##1. Mô tả bài toán

Bài toán nghiên cứu so sánh hiệu quả của ba thuật toán phân cụm dựa trên lưới (grid-based clustering) trong việc phát hiện các vùng tập trung điểm quan tâm (Points of Interest - POI) trên bản đồ địa lý:

| Thuật toán | Mô tả |
|---|---|
| **GCBD** (Grid-based Clustering using Boundary Detection) | Thuật toán gốc theo bài báo Du & Wu (Entropy 2022) |
| **GK-GCBD** (Gaussian Kernel GCBD) | Phiên bản cải tiến, thay hàm membership tuyến tính bằng Gaussian kernel |
| **CLIQUE** | Thuật toán grid-based cổ điển, làm baseline so sánh |

**Mục tiêu:** Đánh giá xem GK-GCBD (phiên bản cải tiến) có vượt trội hơn GCBD gốc và CLIQUE về chất lượng phân cụm và tốc độ xử lý hay không.

---

## 2. Yêu cầu dữ liệu & Link tải

Notebook sử dụng **hai loại dữ liệu**:

### 2.1 Dữ liệu tổng hợp (Synthetic dataset) — dùng cho Benchmark 1
- **Định dạng:** file `.arff` (Attribute-Relation File Format)
- **Nguồn:** Clustering benchmark datasets
- **Link tải:** https://github.com/trisminh123/GCBD/tree/main/Datasets/Synthetic_dataset
- **Đặc điểm:** Có nhãn ground truth (cột `class`/`label`) để tính AMI, FMI
- **Cấu trúc file:**
```
@relation dataset_name
@attribute x REAL
@attribute y REAL
@attribute class {1,2,3,...}
@data
1.2, 3.4, 1
...
```

### 2.2 Dữ liệu thực tế OpenStreetMap (OSM dataset) — dùng cho Benchmark 2 & 3
- **Định dạng:** file `.txt` (tab-separated)
- **Nguồn:** OpenStreetMap (OSM) — dữ liệu điểm quan tâm (POI) tại TP.HCM và các tỉnh thành
- **Link tải OSM:** https://github.com/trisminh123/GCBD/tree/main/Datasets/OSM_dataset
- **Cấu trúc file:**
```
PoiID\tNEAR_X\tNEAR_Y
1\t106.6297\t10.8231
2\t106.7025\t10.7769
...
```

| Cột | Ý nghĩa |
|---|---|
| `PoiID` hoặc `IdPoints` | ID điểm quan tâm |
| `NEAR_X` | Kinh độ (Longitude) |
| `NEAR_Y` | Vĩ độ (Latitude) |

---

##3. Đầu ra mong đợi

| Benchmark | Kết quả đầu ra |
|---|---|
| **Benchmark 1 - Synthetic** | File CSV tổng hợp AMI/FMI; ảnh phân cụm 4 panel (Ground Truth / GCBD / GK-GCBD / CLIQUE) |
| **Benchmark 2 - OSM** | File Excel với chỉ số Silhouette, BSS, WSS, DB, Dunn, CH cho bộ tham số tốt nhất |
| **Benchmark 3 - Runtime** | File Excel thời gian chạy trung bình; biểu đồ đường so sánh tốc độ |


---
#PHẦN A — CÀI ĐẶT & THIẾT LẬP MÔI TRƯỜNG


In [16]:
# ============================================================
# A1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# ============================================================
# pyclustering: cung cấp thuật toán CLIQUE
# openpyxl   : ghi file Excel (.xlsx)
# scipy      : tính khoảng cách, xử lý file .arff
# scikit-learn: chuẩn hóa dữ liệu, tính chỉ số đánh giá

!pip install pyclustering openpyxl scipy scikit-learn --quiet
print('Cài đặt hoàn tất')

✅ Cài đặt hoàn tất


## A2. Lưu ý về cấu trúc dự án

Cấu trúc thư mục sẽ được tạo tự động khi clone GitHub ở **Phần B**:
```
/content/GCBD/
├── Algorithm/             ← Code các thuật toán
├── Datasets/
│   ├── Synthetic_dataset/ ← File .arff
│   └── OSM_dataset/       ← File .txt
└── results/
    ├── Synthetic/         ← Kết quả benchmark synthetic
    ├── OSM/               ← Kết quả benchmark OSM
    └── runtime/           ← Kết quả so sánh tốc độ
```
> ▶️ **Chạy Phần A1 (cài thư viện) rồi chuyển thẳng sang Phần B để clone repo.**


---
# PHẦN B — THU THẬP DỮ LIỆU TỪ GITHUB


## B1. Clone toàn bộ project từ GitHub

Toàn bộ project (code + dataset) đã được lưu tại:
> **https://github.com/trisminh123/GCBD**

Cấu trúc repo sau khi clone:
```
/content/GCBD/
├── Algorithm/                 ← Code thuật toán
│   ├── fastgrid.py
│   ├── fastgrid_gaussian.py
│   ├── link_core.py
│   ├── merge_core_nd.py
│   ├── single_peel.py
│   └── unique_rows.py
├── Datasets/
│   ├── Synthetic_dataset/     ← File .arff
│   └── OSM_dataset/           ← File .txt
├── results/                   ← Kết quả sẽ được lưu tại đây
├── benchmark_synthetic.py
├── benchmark_OSMdataset.py
└── benchmark_runtime.py
```


In [18]:
# ============================================================
# B2. CLONE PROJECT TỪ GITHUB & THIẾT LẬP MÔI TRƯỜNG
# ============================================================
# Clone repo về /content/GCBD, sau đó:
#   - cd vào thư mục để mọi đường dẫn tương đối hoạt động đúng
#   - Tạo thư mục results nếu chưa có (Git không track folder rỗng)

import os

REPO_URL = 'https://github.com/trisminh123/GCBD.git'
REPO_DIR = '/content/GCBD'

# Clone repo (--depth 1: chỉ lấy commit mới nhất, nhanh hơn)
if os.path.exists(REPO_DIR):
    print('  Thư mục đã tồn tại — cập nhật repo...')
    !cd {REPO_DIR} && git pull
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

# Chuyển working directory vào repo để đường dẫn tương đối hoạt động
os.chdir(REPO_DIR)
print(f'\n Working directory: {os.getcwd()}')

# Đảm bảo các thư mục kết quả tồn tại
for d in ['results/Synthetic', 'results/OSM', 'results/runtime']:
    os.makedirs(d, exist_ok=True)

print('\n Clone hoàn tất! Cấu trúc thư mục:')
!find . -maxdepth 3 -not -path './.git/*' | sort


  Thư mục đã tồn tại — cập nhật repo...
Already up to date.

 Working directory: /content/GCBD

 Clone hoàn tất! Cấu trúc thư mục:
.
./Algorithm
./Algorithm/fastgrid_gaussian.py
./Algorithm/fastgrid.py
./Algorithm/link_core.py
./Algorithm/merge_core_nd.py
./Algorithm/__pycache__
./Algorithm/__pycache__/fastgrid.cpython-310.pyc
./Algorithm/__pycache__/fastgrid_gaussian.cpython-310.pyc
./Algorithm/__pycache__/link_core.cpython-310.pyc
./Algorithm/__pycache__/merge_core_nd.cpython-310.pyc
./Algorithm/__pycache__/single_peel.cpython-310.pyc
./Algorithm/__pycache__/unique_rows.cpython-310.pyc
./Algorithm/single_peel.py
./Algorithm/unique_rows.py
./benchmark_OSMdataset.py
./benchmark_runtime.py
./benchmark_synthetic.py
./Datasets
./Datasets/OSM_dataset
./Datasets/OSM_dataset/Acute1_4_CanThoInput.txt
./Datasets/OSM_dataset/Acute1_4_DongNaiInput.txt
./Datasets/OSM_dataset/Acute1_4_Nhom1Input.txt
./Datasets/OSM_dataset/Acute1_4_Nhom2Input.txt
./Datasets/OSM_dataset/Acute1_4_Nhom3Input.txt
./Dat

In [3]:
# ============================================================
# B3. KIỂM TRA DỮ LIỆU ĐÃ CÓ TRONG REPO
# ============================================================

from pathlib import Path

arff_files = sorted(Path('Datasets/Synthetic_dataset').glob('*.arff'))
txt_files  = sorted(Path('Datasets/OSM_dataset').glob('*.txt'))

print(f'Dữ liệu Synthetic (.arff): {len(arff_files)} file')
for f in arff_files:
    print(f'   • {f.name}')

print(f'\nDữ liệu OSM (.txt): {len(txt_files)} file')
for f in txt_files:
    print(f'   • {f.name}')

if not arff_files and not txt_files:
    print('\nChưa thấy dữ liệu — hãy kiểm tra lại repo GitHub đã có file chưa!')
else:
    print('\nDữ liệu sẵn sàng — tiếp tục chạy từ Phần C!')


📊 Dữ liệu Synthetic (.arff): 16 file
   • 3-spiral.arff
   • R15.arff
   • aggregation.arff
   • blobs.arff
   • circle.arff
   • compound.arff
   • dense-disk-5000.arff
   • disk-4000n.arff
   • elliptical_10_2.arff
   • flame.arff
   • insect.arff
   • pathbased.arff
   • rings.arff
   • smile3.arff
   • square4.arff
   • tetra.arff

Dữ liệu OSM (.txt): 5 file
   • Acute1_4_CanThoInput.txt
   • Acute1_4_DongNaiInput.txt
   • Acute1_4_Nhom1Input.txt
   • Acute1_4_Nhom2Input.txt
   • Acute1_4_Nhom3Input.txt

Dữ liệu sẵn sàng — tiếp tục chạy từ Phần C!


---
# PHẦN C — XỬ LÝ DỮ LIỆU


In [19]:
# ============================================================
# C1. ĐỌC & XỬ LÝ DỮ LIỆU SYNTHETIC (.arff)
# ============================================================
#
# Input:
#   - filepath: đường dẫn đến file .arff
#
# Output:
#   - data       : np.ndarray shape (n_samples, n_features) — tọa độ điểm dữ liệu
#   - true_labels: np.ndarray shape (n_samples,) — nhãn ground truth (int), -1 nếu không có
#
# Lưu ý:
#   - Cột nhãn được nhận dạng tự động qua tên: 'class', 'label', 'target', 'cluster'
#   - Nhãn dạng chuỗi (vd: 'c1', 'c2') được ánh xạ sang số nguyên

import numpy as np
from scipy.io.arff import loadarff

def load_arff(filepath: str):
    """
    Đọc file .arff và trả về (data, true_labels).

    Input:
        filepath (str): đường dẫn đến file .arff

    Output:
        data        (np.ndarray): shape (n, m) — tọa độ các điểm dữ liệu
        true_labels (np.ndarray): shape (n,)   — nhãn phân cụm ground truth
    """
    raw, meta = loadarff(filepath)
    feature_cols, label_col = [], None

    for name in meta.names():
        typ = meta[name][0]
        col = np.array([row[name] for row in raw])

        if name.lower() in ('class', 'label', 'target', 'cluster'):
            label_col = col
        elif typ in ('numeric', 'real', 'integer'):
            feature_cols.append(col.astype(float))

    data = np.column_stack(feature_cols) if feature_cols else None

    if label_col is not None:
        try:
            labels = label_col.astype(int)
        except (ValueError, TypeError):
            unique_vals = sorted(set(label_col))
            mapping     = {v: i for i, v in enumerate(unique_vals)}
            labels      = np.array([mapping[v] for v in label_col], dtype=int)
    else:
        labels = np.full(len(data), -1, dtype=int)

    return data, labels


# --- Kiểm tra với file đầu tiên (nếu có) ---
if arff_files:
    sample_data, sample_labels = load_arff(str(arff_files[0]))
    print(f'📋 File: {arff_files[0].name}')
    print(f'   Số điểm dữ liệu  : {sample_data.shape[0]}')
    print(f'   Số chiều          : {sample_data.shape[1]}')
    print(f'   Số cụm ground truth: {len(np.unique(sample_labels[sample_labels >= 0]))}')
    print(f'   Vài dòng đầu:\n{sample_data[:3]}')
else:
    print('⚠️  Chưa có file .arff để kiểm tra')

📋 File: 3-spiral.arff
   Số điểm dữ liệu  : 312
   Số chiều          : 2
   Số cụm ground truth: 3
   Vài dòng đầu:
[[31.95  7.95]
 [31.15  7.3 ]
 [30.45  6.65]]


In [20]:
# ============================================================
# C2. ĐỌC & XỬ LÝ DỮ LIỆU OSM (.txt)
# ============================================================
#
# Input:
#   - filepath: đường dẫn đến file .txt (tab-separated)
#
# Output:
#   - dataset_name: tên dataset (lấy từ tên file)
#   - X           : np.ndarray shape (n_samples, 2) — tọa độ [NEAR_X, NEAR_Y]
#
# Yêu cầu cột:
#   - Cột ID: 'PoiID' hoặc 'IdPoints'
#   - Cột tọa độ: 'NEAR_X' (kinh độ), 'NEAR_Y' (vĩ độ)

import pandas as pd
from pathlib import Path

def load_osm_dataset(filepath: str):
    """
    Đọc file OSM dạng .txt tab-separated.

    Input:
        filepath (str): đường dẫn đến file .txt

    Output:
        dataset_name (str)       : tên dataset (từ tên file)
        X            (np.ndarray): shape (n, 2) — tọa độ [NEAR_X, NEAR_Y]
    """
    df = pd.read_csv(filepath, sep='\t')
    df.columns = [c.strip() for c in df.columns]

    # Tìm cột ID (PoiID hoặc IdPoints)
    id_col = next((c for c in ['PoiID', 'IdPoints'] if c in df.columns), None)
    if id_col is None:
        raise ValueError(f'Không tìm thấy cột PoiID/IdPoints trong {filepath}')

    # Kiểm tra cột tọa độ
    for col in ['NEAR_X', 'NEAR_Y']:
        if col not in df.columns:
            raise ValueError(f'Không tìm thấy cột {col} trong {filepath}')

    X = df[['NEAR_X', 'NEAR_Y']].values.astype(float)
    dataset_name = Path(filepath).stem
    return dataset_name, X


# --- Kiểm tra với file đầu tiên (nếu có) ---
if txt_files:
    name, X_sample = load_osm_dataset(str(txt_files[0]))
    print(f'File: {txt_files[0].name}')
    print(f'   Dataset name      : {name}')
    print(f'   Số điểm POI       : {X_sample.shape[0]}')
    print(f'   Phạm vi NEAR_X    : [{X_sample[:,0].min():.4f}, {X_sample[:,0].max():.4f}]')
    print(f'   Phạm vi NEAR_Y    : [{X_sample[:,1].min():.4f}, {X_sample[:,1].max():.4f}]')
    print(f'   Vài dòng đầu:\n{X_sample[:3]}')
else:
    print('Chưa có file .txt để kiểm tra')

File: Acute1_4_CanThoInput.txt
   Dataset name      : Acute1_4_CanThoInput
   Số điểm POI       : 2563
   Phạm vi NEAR_X    : [533667.0753, 589810.8261]
   Phạm vi NEAR_Y    : [1100902.4610, 1140257.5690]
   Vài dòng đầu:
[[ 586690.528  1110905.697 ]
 [ 586260.5348 1109495.351 ]
 [ 585608.1907 1108898.703 ]]


In [21]:
# ============================================================
# C3. TRỰC QUAN HÓA DỮ LIỆU THÔ
# ============================================================
# Vẽ scatter plot để kiểm tra phân bố dữ liệu trước khi phân cụm

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, min(2, len(txt_files) + len(arff_files)),
                         figsize=(14, 5), squeeze=False)
ax_idx = 0

# Vẽ dữ liệu OSM
if txt_files:
    nm, Xv = load_osm_dataset(str(txt_files[0]))
    axes[0][ax_idx].scatter(Xv[:, 0], Xv[:, 1], s=2, alpha=0.4, color='#2196F3')
    axes[0][ax_idx].set_title(f'OSM: {nm}\n(n={len(Xv)} điểm POI)', fontsize=11)
    axes[0][ax_idx].set_xlabel('Kinh độ (NEAR_X)')
    axes[0][ax_idx].set_ylabel('Vĩ độ (NEAR_Y)')
    ax_idx += 1

# Vẽ dữ liệu Synthetic
if arff_files and ax_idx < axes[0].shape[0]:
    sd, sl = load_arff(str(arff_files[0]))
    unique_l = np.unique(sl)
    colors_gt = plt.cm.tab10(np.linspace(0, 1, len(unique_l)))
    for lbl, c in zip(unique_l, colors_gt):
        mask = sl == lbl
        axes[0][ax_idx].scatter(sd[mask, 0], sd[mask, 1], s=5, alpha=0.6,
                                 color=c, label=f'Cluster {lbl}')
    axes[0][ax_idx].set_title(f'Synthetic: {arff_files[0].stem}\n(n={len(sd)} điểm)', fontsize=11)
    axes[0][ax_idx].set_xlabel('X')
    axes[0][ax_idx].set_ylabel('Y')
    axes[0][ax_idx].legend(fontsize=7, loc='best')

plt.suptitle('Phân bố dữ liệu thô (trước khi phân cụm)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/raw_data_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print('Đã lưu hình: results/raw_data_visualization.png')

Đã lưu hình: results/raw_data_visualization.png


---
# PHẦN D — CÀI ĐẶT CÁC THUẬT TOÁN

> Mỗi hàm được ghi rõ **Input**, **Output** và giải thích từng bước.


## D1. Hàm hỗ trợ


In [22]:
# ============================================================
# D1a. unique_rows — Tìm các hàng duy nhất trong ma trận
# ============================================================
#
# Input:
#   - a (np.ndarray): ma trận 2D shape (n, m)
#
# Output:
#   - c      (np.ndarray): các hàng duy nhất (đã sắp xếp), shape (k, m)
#   - ind_a  (np.ndarray): index trong a của các hàng duy nhất
#   - ind_c  (np.ndarray): với mỗi hàng của a, index (1-based) trong c
#
# Vai trò trong GCBD:
#   Xác định tập các node lưới thực sự tồn tại (sparse nodes)
#   từ danh sách toàn bộ corner nodes (có thể trùng lặp)

%%writefile Algorithm/unique_rows.py
import numpy as np

def unique_rows(a: np.ndarray):
    """
    Tìm các hàng duy nhất của ma trận a.

    Input:
        a (np.ndarray): ma trận shape (n, m)

    Output:
        c    (np.ndarray): các hàng duy nhất, shape (k, m)
        ind_a(np.ndarray): index trong a của các hàng duy nhất
        ind_c(np.ndarray): với mỗi hàng của a, index 1-based trong c
    """
    ind_sort = np.lexsort(a.T[::-1])
    sort_a   = a[ind_sort]

    diff = np.any(sort_a[:-1] != sort_a[1:], axis=1)
    mask = np.concatenate([[True], diff])

    c     = sort_a[mask]
    ind_a = ind_sort[mask]

    ind_c_sorted = np.cumsum(mask)           # 1-based trong thứ tự đã sort
    ind_c = np.empty(a.shape[0], dtype=int)
    ind_c[ind_sort] = ind_c_sorted

    return c, ind_a, ind_c

Overwriting Algorithm/unique_rows.py


In [23]:
# ============================================================
# D1b. single_peel — Loại bỏ node mật độ thấp (1 vòng lặp)
# ============================================================
#
# Input:
#   - iters               (int)       : số vòng lặp hiện tại (bắt đầu từ 1)
#   - percentile          (float)     : tỉ lệ node bị loại mỗi vòng (vd: 0.1 = 10%)
#   - node_sparse         (np.ndarray): tọa độ các node, shape (N, m)
#   - density             (np.ndarray): mật độ từng node, shape (N,)
#   - activate_node_sparse(np.ndarray): cờ hoạt động (1=còn, 0=đã loại), shape (N,)
#   - old_unactivate_nodes(np.ndarray): index các node đã bị loại vòng trước
#
# Output:
#   - activate_node_sparse(np.ndarray): cờ hoạt động đã cập nhật
#   - old_unactivate_nodes(np.ndarray): tập node bị loại tích lũy
#   - new_unactivate_nodes(np.ndarray): các node mới bị loại vòng này
#
# Công thức ngưỡng (Eq.8 trong paper):
#   index = ceil(N × (1 − (1−p)^t)) − 1
#   → node có mật độ ≤ density[index] sẽ bị loại

%%writefile Algorithm/single_peel.py
import numpy as np

def single_peel(iters, percentile, node_sparse, density,
                activate_node_sparse, old_unactivate_nodes):
    """
    Loại bỏ các node có mật độ thấp (boundary detection - 1 vòng).

    Input:
        iters               (int)        : vòng lặp hiện tại (từ 1)
        percentile          (float)      : tỉ lệ peel mỗi vòng
        node_sparse         (np.ndarray) : tọa độ các node, shape (N, m)
        density             (np.ndarray) : mật độ, shape (N,)
        activate_node_sparse(np.ndarray) : cờ hoạt động, shape (N,)
        old_unactivate_nodes(np.ndarray) : node đã bị loại vòng trước

    Output:
        activate_node_sparse (np.ndarray): cờ đã cập nhật
        old_unactivate_nodes (np.ndarray): tập node bị loại tích lũy
        new_unactivate_nodes (np.ndarray): node mới bị loại vòng này
    """
    no_node_sparse = node_sparse.shape[0]

    # Sắp xếp node theo mật độ tăng dần
    order_by_density    = np.argsort(density)
    nodes_sorted_by_dens = density[order_by_density]

    # Tính ngưỡng theo công thức tích lũy
    index_percentile = int(np.ceil(no_node_sparse * (1 - (1 - percentile) ** iters))) - 1
    index_percentile = max(0, min(index_percentile, no_node_sparse - 1))
    threshold_value  = nodes_sorted_by_dens[index_percentile]

    # Xác định node bị loại
    unactivate_nodes     = np.where(density <= threshold_value)[0]
    new_unactivate_nodes = np.setdiff1d(unactivate_nodes, old_unactivate_nodes)
    old_unactivate_nodes = unactivate_nodes.copy()

    # Cập nhật cờ hoạt động
    activate_node_sparse[new_unactivate_nodes] = 0

    return activate_node_sparse, old_unactivate_nodes, new_unactivate_nodes

Overwriting Algorithm/single_peel.py


In [24]:
# ============================================================
# D1c. link_core — Liên kết boundary node với core node
# ============================================================
#
# Input:
#   - dist_node            (np.ndarray): ma trận khoảng cách Chebyshev giữa các node
#   - node_ind_point_cell  (list)      : node_ind_point_cell[j] = index các điểm lân cận node j
#   - activate_node_sparse (np.ndarray): cờ hoạt động node, shape (N,)
#   - new_unactivate_nodes (np.ndarray): index các node vừa bị peel
#   - density              (np.ndarray): mật độ node, shape (N,)
#
# Output:
#   - link_core_nodes_indices (np.ndarray): với mỗi boundary node, index core node gần nhất
#   - unactivate_points       (np.ndarray): index điểm dữ liệu bị loại khỏi density
#
# Nguyên tắc: boundary node được liên kết đến core node
#   có khoảng cách gần nhất và (nếu bằng nhau) mật độ cao nhất

%%writefile Algorithm/link_core.py
import numpy as np

def link_core(dist_node, node_ind_point_cell,
              activate_node_sparse, new_unactivate_nodes, density):
    """
    Liên kết các boundary node với core node gần nhất có mật độ cao nhất.

    Input:
        dist_node            (np.ndarray): ma trận khoảng cách Chebyshev, shape (N, N)
        node_ind_point_cell  (list)      : list N phần tử, mỗi phần tử là array index điểm
        activate_node_sparse (np.ndarray): cờ hoạt động, shape (N,)
        new_unactivate_nodes (np.ndarray): boundary node mới bị peel
        density              (np.ndarray): mật độ node, shape (N,)

    Output:
        link_core_nodes_indices (np.ndarray): core node liên kết với mỗi boundary node
        unactivate_points       (np.ndarray): điểm dữ liệu bị loại
    """
    unactivate_points = np.array([], dtype=int)

    # Chỉ xét khoảng cách đến các node đang active
    dist = dist_node[:, new_unactivate_nodes] * activate_node_sparse[:, np.newaxis]
    dist[dist == 0] = np.inf     # inactive node → khoảng cách vô cực

    min_dist = np.min(dist, axis=0)
    link_core_nodes_indices = np.zeros(len(new_unactivate_nodes), dtype=int)

    for i, node_idx in enumerate(new_unactivate_nodes):
        # Gom toàn bộ điểm dữ liệu thuộc boundary node này vào danh sách loại
        unactivate_points = np.union1d(unactivate_points,
                                       node_ind_point_cell[node_idx])
        # Trong số các core node có khoảng cách tối thiểu, chọn node mật độ cao nhất
        min_dist_indices = np.where(dist[:, i] == min_dist[i])[0]
        sec_ind = np.argmax(density[min_dist_indices])
        link_core_nodes_indices[i] = min_dist_indices[sec_ind]

    return link_core_nodes_indices, unactivate_points

Overwriting Algorithm/link_core.py


In [25]:
# ============================================================
# D1d. merge_core_nd — Gộp core nodes thành các cluster
# ============================================================
#
# Input:
#   - dist_last (np.ndarray): ma trận khoảng cách (inactive node đã được set = inf)
#   - core_nd   (np.ndarray): index (0-based) của các core node còn lại
#
# Output:
#   - clusters (list of np.ndarray): danh sách các cụm,
#     mỗi cụm chứa index (0-based) của các node thuộc cụm đó
#
# Nguyên tắc (Definition 6 trong paper):
#   Hai core node là "liên kết" nếu khoảng cách Chebyshev = 1
#   → Gộp theo thành phần liên thông (connected components)

%%writefile Algorithm/merge_core_nd.py
import numpy as np
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

def merge_core_nd(dist_last: np.ndarray, core_nd: np.ndarray) -> list:
    """
    Gộp các core node thành cluster dựa trên kết nối lưới.

    Input:
        dist_last (np.ndarray): ma trận khoảng cách, inactive node = inf
        core_nd   (np.ndarray): index 0-based các core node

    Output:
        clusters (list): mỗi phần tử là np.ndarray chứa index các node trong cụm
    """
    # Lấy ma trận khoảng cách riêng giữa các core node
    dist_core = dist_last[np.ix_(core_nd, core_nd)]

    # Hai node liên kết nếu khoảng cách Chebyshev = 1 (Definition 6)
    link = csr_matrix((dist_core == 1).astype(float))

    # Tìm thành phần liên thông → mỗi thành phần là một cluster
    n_components, labels = connected_components(link, directed=False)

    clusters = [core_nd[labels == i] for i in range(n_components)]
    return clusters

Overwriting Algorithm/merge_core_nd.py


## D2. Thuật toán GCBD gốc


In [26]:
# ============================================================
# D2. GCBD — Grid-Based Clustering using Boundary Detection (gốc)
# ============================================================
# Cài đặt theo bài báo: Du & Wu (Entropy 2022)
#
# Pipeline 6 bước:
#   Step 1: Chuẩn hóa dữ liệu → hệ tọa độ lưới (Eq.3)
#   Step 2: Tính mật độ node (Eq.4–5, linear membership)
#   Step 3: Boundary detection bằng peeling lặp nhiều vòng
#   Step 4: Gộp core node thành cluster (connected components)
#   Step 5: Gán boundary node vào cluster tương ứng
#   Step 6: Gán điểm dữ liệu vào cluster
#
# Input:
#   - data       (np.ndarray): shape (n, m) — dữ liệu đầu vào
#   - no_grid    (int)       : số interval mỗi chiều (l trong paper), mặc định 26
#   - percentile (float)     : tỉ lệ peel mỗi vòng, mặc định 0.1
#   - max_iters  (int)       : số vòng lặp tối đa (T trong paper), mặc định 9
#
# Output:
#   - pt_clusters (np.ndarray): shape (n,) — nhãn cluster cho mỗi điểm dữ liệu
#     (nhãn > 0 là điểm được phân cụm; nhãn = 0 là noise)

%%writefile Algorithm/fastgrid.py
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix

from Algorithm.unique_rows import unique_rows
from Algorithm.single_peel import single_peel
from Algorithm.link_core import link_core
from Algorithm.merge_core_nd import merge_core_nd


def fastgrid(
    data: np.ndarray,
    no_grid: int = 26,
    percentile: float = 0.1,
    max_iters: int = 9,
) -> np.ndarray:
    """
    GCBD: Grid-Based Clustering using Boundary Detection (Du & Wu, Entropy 2022).

    Input:
        data       (np.ndarray): shape (n, m) — dữ liệu đầu vào
        no_grid    (int)       : số interval mỗi chiều (l), mặc định 26
        percentile (float)     : tỉ lệ peel mỗi vòng, mặc định 0.1
        max_iters  (int)       : số vòng lặp tối đa (T), mặc định 9

    Output:
        pt_clusters (np.ndarray): shape (n,) — nhãn cluster (>0) hoặc noise (=0)
    """
    nd_change_thres = 1
    no_objs, no_dims = data.shape

    # ── STEP 1: Chuẩn hóa về hệ tọa độ lưới (Eq.3) ──────────────────────────
    # Φ(xij) = l * (xij - xmin) / (xmax - xmin) + 1 ∈ [1, l+1]
    data_min = data.min(axis=0)
    data_max = data.max(axis=0)
    data_scaled = (no_grid - 1) * (data - data_min) / (data_max - data_min) + 1
    data_temp = data_scaled.copy()
    data_temp[data_temp == no_grid] = no_grid - 1e-10

    # ── STEP 2: Tính mật độ node (Eq.4–5) ────────────────────────────────────
    n_corners    = 2 ** no_dims
    bin_sequence = np.array(
        [[int(b) for b in format(k, f'0{no_dims}b')] for k in range(n_corners)],
        dtype=float,
    )

    data_floor     = np.floor(data_temp)
    nodes_exp      = data_floor[:, np.newaxis, :] + bin_sequence[np.newaxis, :, :]
    data_nodes_mat = nodes_exp.reshape(no_objs * n_corners, no_dims)

    # Eq.(4): fj(vj) = max(1 - |Φ(xij) - vj|, 0)
    data_exp         = data_temp[:, np.newaxis, :]
    fd               = np.maximum(1.0 - np.abs(data_exp - nodes_exp), 0.0)
    # Eq.(5): ρv = Π fj(vj)
    single_gain_mat  = np.prod(fd, axis=2)
    single_gain_onedim = single_gain_mat.reshape(no_objs * n_corners)

    node_sparse, _, IC = unique_rows(data_nodes_mat)
    no_node_sparse     = node_sparse.shape[0]

    rows = np.arange(no_objs * n_corners)
    cols = IC - 1
    membership_remat = csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(no_objs * n_corners, no_node_sparse),
    )
    density = np.array(membership_remat.T.dot(single_gain_onedim)).ravel()

    membership_dense    = membership_remat.toarray()
    membership_pt       = membership_dense.reshape(no_objs, n_corners, no_node_sparse).max(axis=1)
    node_ind_point_cell = [np.where(membership_pt[:, j] > 0)[0] for j in range(no_node_sparse)]

    # ── STEP 3: Boundary Detection (peeling lặp) ─────────────────────────────
    dist_node            = squareform(pdist(node_sparse, metric='chebyshev'))
    activate_node_sparse = np.ones(no_node_sparse)
    activate_data_remat  = np.ones((n_corners, no_objs))
    old_unactivate_nodes = np.array([], dtype=int)
    iters, prev_iter_nd_len = 1, 0
    border_nodes_per_iter, core_nodes_per_iter = [], []

    while max_iters >= iters:
        activate_node_sparse, old_unactivate_nodes, new_add_unactivate_nodes = single_peel(
            iters, percentile, node_sparse, density,
            activate_node_sparse, old_unactivate_nodes,
        )
        if len(new_add_unactivate_nodes) == 0:
            break
        curr_iter_nd_len = len(old_unactivate_nodes)
        if abs(curr_iter_nd_len - prev_iter_nd_len) < nd_change_thres and iters != 1:
            break
        prev_iter_nd_len = curr_iter_nd_len

        link_core_nodes_indices, unactivate_points = link_core(
            dist_node, node_ind_point_cell,
            activate_node_sparse, new_add_unactivate_nodes, density,
        )
        border_nodes_per_iter.append(new_add_unactivate_nodes.copy())
        core_nodes_per_iter.append(link_core_nodes_indices.copy())

        # Cập nhật density (Eq.9): loại bỏ điểm đã bị peel
        activate_data_remat[:, unactivate_points] = 0
        oneiter_gain    = single_gain_mat * activate_data_remat.T
        oneiter_gain_1d = oneiter_gain.reshape(no_objs * n_corners)
        density = np.array(membership_remat.T.dot(oneiter_gain_1d)).ravel()
        iters += 1

    # ── STEP 4: Gộp core node thành cluster ──────────────────────────────────
    core_nd   = np.setdiff1d(np.arange(no_node_sparse), old_unactivate_nodes)
    dist_last = (dist_node * activate_node_sparse[:, np.newaxis]).T
    dist_last[dist_last == 0] = np.inf

    cluster_lists = merge_core_nd(dist_last, core_nd)
    nd_clusters   = np.zeros(no_node_sparse, dtype=int)
    for cluster_index, cl in enumerate(cluster_lists, start=1):
        nd_clusters[cl] = cluster_index

    # ── STEP 5: Gán boundary node vào cluster ────────────────────────────────
    for i in range(len(border_nodes_per_iter) - 1, -1, -1):
        nd_clusters[border_nodes_per_iter[i]] = nd_clusters[core_nodes_per_iter[i]]

    # ── STEP 6: Gán điểm dữ liệu vào cluster ────────────────────────────────
    nearest_node      = np.round(data_scaled).astype(int)
    node_sparse_int   = node_sparse.astype(int)
    node_coord_to_idx = {tuple(node_sparse_int[j]): j for j in range(no_node_sparse)}
    locb = np.array(
        [node_coord_to_idx.get(tuple(nearest_node[i]), 0) for i in range(no_objs)],
        dtype=int,
    )
    return nd_clusters[locb]

Overwriting Algorithm/fastgrid.py


## D3. Thuật toán GK-GCBD (cải tiến Gaussian Kernel)


In [27]:
# ============================================================
# D3. GK-GCBD — GCBD với Gaussian Kernel (phiên bản cải tiến)
# ============================================================
#
# Khác biệt duy nhất so với GCBD gốc: STEP 2
#
# GCBD gốc (Eq.4-5):
#   ρ_v = Π max(1 - |Φ(x_ij) - v_j|, 0)
#   → Linear membership, gãy khúc tại biên cell
#
# GK-GCBD (Gaussian kernel):
#   ρ_v = Σ exp(-||Φ(x_i) - v||² / 2σ²)
#   → Smooth density surface, gradient mượt hơn
#   → Boundary detection chính xác hơn
#
# Input:
#   - data       (np.ndarray): shape (n, m) — dữ liệu đầu vào
#   - no_grid    (int)       : số interval mỗi chiều, mặc định 26
#   - percentile (float)     : tỉ lệ peel mỗi vòng, mặc định 0.1
#   - max_iters  (int)       : số vòng lặp tối đa, mặc định 9
#   - sigma      (float)     : độ rộng Gaussian kernel (σ), mặc định 1.0
#
# Output:
#   - pt_clusters (np.ndarray): shape (n,) — nhãn cluster (>0) hoặc noise (=0)

%%writefile Algorithm/fastgrid_gaussian.py
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix

from Algorithm.unique_rows import unique_rows
from Algorithm.single_peel import single_peel
from Algorithm.link_core import link_core
from Algorithm.merge_core_nd import merge_core_nd


def fastgrid_gaussian(
    data      : np.ndarray,
    no_grid   : int   = 26,
    percentile: float = 0.1,
    max_iters : int   = 9,
    sigma     : float = 1.0,
) -> np.ndarray:
    """
    GK-GCBD: GCBD với Gaussian Kernel density estimation.

    Input:
        data       (np.ndarray): shape (n, m) — dữ liệu đầu vào
        no_grid    (int)       : số interval mỗi chiều, mặc định 26
        percentile (float)     : tỉ lệ peel mỗi vòng, mặc định 0.1
        max_iters  (int)       : số vòng lặp tối đa, mặc định 9
        sigma      (float)     : bandwidth Gaussian kernel (σ), mặc định 1.0

    Output:
        pt_clusters (np.ndarray): shape (n,) — nhãn cluster (>0) hoặc noise (=0)
    """
    nd_change_thres = 1
    no_objs, no_dims = data.shape

    # ── STEP 1: Chuẩn hóa (giữ nguyên) ──────────────────────────────────────
    data_min    = data.min(axis=0)
    data_max    = data.max(axis=0)
    data_scaled = (no_grid - 1) * (data - data_min) / (data_max - data_min) + 1
    data_temp   = data_scaled.copy()
    data_temp[data_temp == no_grid] = no_grid - 1e-10

    # ── STEP 2: Gaussian Kernel density (THAY ĐỔI CHÍNH) ─────────────────────
    # ρ_v = Σ_{x_i ∈ neighbors(v)} exp(-||Φ(x_i) - v||² / 2σ²)
    n_corners    = 2 ** no_dims
    bin_sequence = np.array(
        [[int(b) for b in format(k, f'0{no_dims}b')] for k in range(n_corners)],
        dtype=float,
    )

    data_floor     = np.floor(data_temp)
    nodes_exp      = data_floor[:, np.newaxis, :] + bin_sequence[np.newaxis, :, :]
    data_nodes_mat = nodes_exp.reshape(no_objs * n_corners, no_dims)

    # Tính khoảng cách Euclidean bình phương từ điểm đến corner node
    data_exp       = data_temp[:, np.newaxis, :]
    diff           = data_exp - nodes_exp
    sq_dist        = np.sum(diff ** 2, axis=2)

    # Gaussian weight thay thế linear membership
    single_gain_mat    = np.exp(-sq_dist / (2.0 * sigma ** 2))
    single_gain_onedim = single_gain_mat.reshape(no_objs * n_corners)

    node_sparse, _, IC = unique_rows(data_nodes_mat)
    no_node_sparse     = node_sparse.shape[0]

    rows = np.arange(no_objs * n_corners)
    cols = IC - 1
    membership_remat = csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(no_objs * n_corners, no_node_sparse),
    )
    density = np.array(membership_remat.T.dot(single_gain_onedim)).ravel()

    membership_dense    = membership_remat.toarray()
    membership_pt       = membership_dense.reshape(no_objs, n_corners, no_node_sparse).max(axis=1)
    node_ind_point_cell = [np.where(membership_pt[:, j] > 0)[0] for j in range(no_node_sparse)]

    # ── STEP 3–6: Giữ nguyên hoàn toàn so với GCBD gốc ──────────────────────
    dist_node            = squareform(pdist(node_sparse, metric='chebyshev'))
    activate_node_sparse = np.ones(no_node_sparse)
    activate_data_remat  = np.ones((n_corners, no_objs))
    old_unactivate_nodes = np.array([], dtype=int)
    iters, prev_iter_nd_len = 1, 0
    border_nodes_per_iter, core_nodes_per_iter = [], []

    while max_iters >= iters:
        activate_node_sparse, old_unactivate_nodes, new_add_unactivate_nodes = single_peel(
            iters, percentile, node_sparse, density,
            activate_node_sparse, old_unactivate_nodes,
        )
        if len(new_add_unactivate_nodes) == 0:
            break
        curr_iter_nd_len = len(old_unactivate_nodes)
        if abs(curr_iter_nd_len - prev_iter_nd_len) < nd_change_thres and iters != 1:
            break
        prev_iter_nd_len = curr_iter_nd_len

        link_core_nodes_indices, unactivate_points = link_core(
            dist_node, node_ind_point_cell,
            activate_node_sparse, new_add_unactivate_nodes, density,
        )
        border_nodes_per_iter.append(new_add_unactivate_nodes.copy())
        core_nodes_per_iter.append(link_core_nodes_indices.copy())

        activate_data_remat[:, unactivate_points] = 0
        oneiter_gain    = single_gain_mat * activate_data_remat.T
        oneiter_gain_1d = oneiter_gain.reshape(no_objs * n_corners)
        density = np.array(membership_remat.T.dot(oneiter_gain_1d)).ravel()
        iters += 1

    core_nd   = np.setdiff1d(np.arange(no_node_sparse), old_unactivate_nodes)
    dist_last = (dist_node * activate_node_sparse[:, np.newaxis]).T
    dist_last[dist_last == 0] = np.inf

    cluster_lists = merge_core_nd(dist_last, core_nd)
    nd_clusters   = np.zeros(no_node_sparse, dtype=int)
    for cluster_index, cl in enumerate(cluster_lists, start=1):
        nd_clusters[cl] = cluster_index

    for i in range(len(border_nodes_per_iter) - 1, -1, -1):
        nd_clusters[border_nodes_per_iter[i]] = nd_clusters[core_nodes_per_iter[i]]

    nearest_node      = np.round(data_scaled).astype(int)
    node_sparse_int   = node_sparse.astype(int)
    node_coord_to_idx = {tuple(node_sparse_int[j]): j for j in range(no_node_sparse)}
    locb = np.array(
        [node_coord_to_idx.get(tuple(nearest_node[i]), 0) for i in range(no_objs)],
        dtype=int,
    )
    return nd_clusters[locb]

Overwriting Algorithm/fastgrid_gaussian.py


In [28]:
# ============================================================
# D4. Kiểm tra nhanh các thuật toán (sanity check)
# ============================================================
# Tạo dữ liệu giả đơn giản, chạy thử để xác nhận không có lỗi

from Algorithm.fastgrid import fastgrid
from Algorithm.fastgrid_gaussian import fastgrid_gaussian

# Tạo dữ liệu 2D đơn giản: 3 cụm rõ ràng
np.random.seed(42)
test_data = np.vstack([
    np.random.randn(50, 2) + [0, 0],
    np.random.randn(50, 2) + [5, 5],
    np.random.randn(50, 2) + [10, 0],
])

# Chạy GCBD gốc
labels_gcbd = fastgrid(test_data, no_grid=20, percentile=0.1, max_iters=5)
print(f'GCBD gốc     : {len(np.unique(labels_gcbd[labels_gcbd>0]))} cụm tìm được')

# Chạy GK-GCBD
labels_gk = fastgrid_gaussian(test_data, no_grid=20, percentile=0.1, max_iters=5, sigma=0.5)
print(f'GK-GCBD      : {len(np.unique(labels_gk[labels_gk>0]))} cụm tìm được')

# Vẽ kết quả kiểm tra
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (lbls, title) in zip(axes, [(labels_gcbd, 'GCBD gốc'), (labels_gk, 'GK-GCBD')]):
    ax.scatter(test_data[:, 0], test_data[:, 1],
               c=lbls, cmap='tab10', s=20, alpha=0.8)
    ax.set_title(f'{title} — Sanity Check', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
print('\n Tất cả thuật toán hoạt động bình thường!')

GCBD gốc     : 3 cụm tìm được
GK-GCBD      : 3 cụm tìm được

Tất cả thuật toán hoạt động bình thường!


---
#  PHẦN E — BENCHMARK 1: DỮ LIỆU TỔNG HỢP (SYNTHETIC)

**Mục tiêu:** So sánh chất lượng phân cụm trên dữ liệu có nhãn ground truth  
**Chỉ số đánh giá:** AMI (Adjusted Mutual Information), FMI (Fowlkes-Mallows Index)  
**Phương pháp:** Grid search toàn bộ tổ hợp tham số, chọn bộ tham số tốt nhất


In [29]:
# ============================================================
# E1. CÀI ĐẶT HÀM ĐÁNH GIÁ VÀ TIỆN ÍCH CHO BENCHMARK SYNTHETIC
# ============================================================

import warnings
import itertools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_mutual_info_score, fowlkes_mallows_score
from sklearn.preprocessing import MinMaxScaler

try:
    from pyclustering.cluster.clique import clique
    CLIQUE_AVAILABLE = True
    print('pyclustering (CLIQUE) sẵn sàng')
except ImportError:
    CLIQUE_AVAILABLE = False
    print(' pyclustering chưa cài — CLIQUE sẽ bị bỏ qua')

# ── Tham số grid search ──────────────────────────────────────────────────────
GRID_PARAMS = {
    'no_grid'   : list(range(6, 52)),      # l = 6 → 51
    'percentile': [0.1],
    'max_iters' : list(range(2, 13)),      # T = 2 → 12
}
SIGMA_VALUES  = [0.25, 0.5, 0.75, 1.0]
CLIQUE_PARAMS = {
    'intervals': list(range(5, 51)),       # l = 5 → 50
    'threshold': list(range(0, 6)),        # c = 0 → 5
}

# ── Màu sắc cluster ──────────────────────────────────────────────────────────
CLUSTER_COLORS = ['#4CAF50','#F44336','#FF9800','#2196F3','#9C27B0',
                  '#795548','#009688','#FF5722','#607D8B','#E91E63',
                  '#CDDC39','#00BCD4']
NOISE_COLOR = '#BDBDBD'


def compute_metrics(true_labels, pred_labels):
    """
    Tính AMI và FMI.

    Input:
        true_labels (np.ndarray): nhãn ground truth
        pred_labels (np.ndarray): nhãn dự đoán

    Output:
        ami (float): Adjusted Mutual Information [−1, 1] (cao hơn = tốt hơn)
        fmi (float): Fowlkes-Mallows Index [0, 1] (cao hơn = tốt hơn)
    """
    mask = true_labels >= 0
    if mask.sum() == 0:
        return None, None
    ami = adjusted_mutual_info_score(true_labels[mask], pred_labels[mask])
    fmi = fowlkes_mallows_score(true_labels[mask], pred_labels[mask])
    return round(ami, 4), round(fmi, 4)


def clique_predict_synthetic(data, intervals, threshold):
    """
    Chạy CLIQUE và trả về nhãn dạng numpy (−1 = noise).

    Input:
        data      (np.ndarray): dữ liệu, shape (n, m)
        intervals (int)       : số khoảng chia mỗi chiều
        threshold (int)       : ngưỡng mật độ tối thiểu

    Output:
        labels (np.ndarray): nhãn cluster (>=0) hoặc noise (=-1)
    """
    if not CLIQUE_AVAILABLE:
        return None
    scaler      = MinMaxScaler()
    data_scaled = scaler.fit_transform(data).tolist()
    inst        = clique(data_scaled, intervals, threshold)
    inst.process()
    clusters = inst.get_clusters()
    labels   = np.full(len(data), -1, dtype=int)
    for cid, indices in enumerate(clusters):
        for idx in indices:
            labels[idx] = cid
    return labels


def labels_to_colors(labels):
    return [NOISE_COLOR if l < 0 else CLUSTER_COLORS[l % len(CLUSTER_COLORS)]
            for l in labels]

print('Hàm đánh giá Benchmark Synthetic đã sẵn sàng')

pyclustering (CLIQUE) sẵn sàng
Hàm đánh giá Benchmark Synthetic đã sẵn sàng


In [30]:
# ============================================================
# E2. CHẠY BENCHMARK SYNTHETIC — GRID SEARCH & ĐÁNH GIÁ
# ============================================================
#
# Quy trình:
#   1. Với mỗi file .arff:
#      a) Chạy grid search GCBD gốc  → chọn bộ tham số có AMI cao nhất
#      b) Chạy grid search GK-GCBD   → chọn bộ tham số + sigma tốt nhất
#      c) Chạy grid search CLIQUE     → chọn intervals + threshold tốt nhất
#      d) Vẽ ảnh so sánh 4 panel
#   2. Xuất CSV tổng hợp kết quả

import os
import pandas as pd

SYNTHETIC_RESULT_DIR = 'results/Synthetic'
os.makedirs(SYNTHETIC_RESULT_DIR, exist_ok=True)

all_rows, summary_rows = [], []

n_combos_orig    = len(GRID_PARAMS['no_grid']) * len(GRID_PARAMS['percentile']) * len(GRID_PARAMS['max_iters'])
n_combos_gaussian = n_combos_orig * len(SIGMA_VALUES)
n_combos_clique   = len(CLIQUE_PARAMS['intervals']) * len(CLIQUE_PARAMS['threshold'])

print(f'Tổng tổ hợp mỗi dataset: GCBD={n_combos_orig}, GK-GCBD={n_combos_gaussian}, CLIQUE={n_combos_clique}')
print('='*65)

arff_files_list = sorted(Path('Datasets/Synthetic_dataset').glob('*.arff'))
if not arff_files_list:
    print('Không tìm thấy file .arff — hãy upload dữ liệu ở Phần B')
else:
    for arff_path in arff_files_list:
        name = arff_path.stem
        print(f'\n[{name}]')
        data, true_labels = load_arff(str(arff_path))

        best_pred = {'orig': None, 'gauss': None, 'clique': None}
        best_ami  = {'orig': -999, 'gauss': -999, 'clique': -999}

        # ── GCBD gốc ──────────────────────────────────────────────────────────
        print(f'  GCBD gốc ({n_combos_orig} combos)...', end=' ', flush=True)
        for combo in itertools.product(*GRID_PARAMS.values()):
            params = dict(zip(GRID_PARAMS.keys(), combo))
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    pred = fastgrid(data, **params)
            except Exception:
                continue
            ami, fmi = compute_metrics(true_labels, pred)
            if ami is None: continue
            all_rows.append({'dataset': name, 'algorithm': 'GCBD_original',
                             **params, 'sigma': '-', 'AMI': ami, 'FMI': fmi})
            if ami > best_ami['orig']:
                best_ami['orig'] = ami
                best_pred['orig'] = pred
                summary_rows.append({'dataset': name, 'algorithm': 'GCBD_original',
                                     **params, 'sigma': '-', 'AMI': ami, 'FMI': fmi})
        print(f'best AMI={best_ami["orig"]:.4f}')

        # ── GK-GCBD ───────────────────────────────────────────────────────────
        for sigma in SIGMA_VALUES:
            print(f'  GK-GCBD σ={sigma} ({n_combos_orig} combos)...', end=' ', flush=True)
            best_sigma_ami = -999
            for combo in itertools.product(*GRID_PARAMS.values()):
                params = dict(zip(GRID_PARAMS.keys(), combo))
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        pred = fastgrid_gaussian(data, **params, sigma=sigma)
                except Exception:
                    continue
                ami, fmi = compute_metrics(true_labels, pred)
                if ami is None: continue
                all_rows.append({'dataset': name, 'algorithm': f'GCBD_gaussian_s{sigma}',
                                 **params, 'sigma': sigma, 'AMI': ami, 'FMI': fmi})
                if ami > best_ami['gauss']:
                    best_ami['gauss'] = ami
                    best_pred['gauss'] = pred
                if ami > best_sigma_ami:
                    best_sigma_ami = ami
                    summary_rows.append({'dataset': name, 'algorithm': f'GCBD_gaussian_s{sigma}',
                                         **params, 'sigma': sigma, 'AMI': ami, 'FMI': fmi})
            print(f'best AMI={best_sigma_ami:.4f}')

        # ── CLIQUE ────────────────────────────────────────────────────────────
        if CLIQUE_AVAILABLE:
            print(f'  CLIQUE ({n_combos_clique} combos)...', end=' ', flush=True)
            for intervals in CLIQUE_PARAMS['intervals']:
                for threshold in CLIQUE_PARAMS['threshold']:
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore')
                            pred = clique_predict_synthetic(data, intervals, threshold)
                        if pred is None: continue
                    except Exception:
                        continue
                    ami, fmi = compute_metrics(true_labels, pred)
                    if ami is None: continue
                    all_rows.append({'dataset': name, 'algorithm': 'CLIQUE',
                                     'intervals': intervals, 'threshold': threshold,
                                     'AMI': ami, 'FMI': fmi})
                    if ami > best_ami['clique']:
                        best_ami['clique'] = ami
                        best_pred['clique'] = pred
                        summary_rows.append({'dataset': name, 'algorithm': 'CLIQUE',
                                             'intervals': intervals, 'threshold': threshold,
                                             'AMI': ami, 'FMI': fmi})
            print(f'best AMI={best_ami["clique"]:.4f}')

        # ── Vẽ ảnh 4 panel ────────────────────────────────────────────────────
        if all(v is not None for v in best_pred.values()):
            fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='white')
            titles = ['Ground Truth', 'GCBD gốc', 'GK-GCBD', 'CLIQUE']
            preds  = [true_labels, best_pred['orig'], best_pred['gauss'], best_pred['clique']]
            for ax, title, lbls in zip(axes, titles, preds):
                ax.scatter(data[:, 0], data[:, 1],
                           c=labels_to_colors(lbls), s=8, alpha=0.85, linewidths=0)
                ax.set_title(title, fontsize=11, fontweight='bold')
                ax.set_xticks([]); ax.set_yticks([])
            plt.suptitle(f'Dataset: {name}', fontsize=13)
            plt.tight_layout()
            fig_path = f'{SYNTHETIC_RESULT_DIR}/clustering_{name}.png'
            plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            print(f'  → Đã lưu hình: {fig_path}')

    # ── Xuất CSV ──────────────────────────────────────────────────────────────
    pd.DataFrame(all_rows).to_csv(f'{SYNTHETIC_RESULT_DIR}/benchmark_results.csv', index=False)
    pd.DataFrame(summary_rows).to_csv(f'{SYNTHETIC_RESULT_DIR}/benchmark_summary.csv', index=False)
    print(f'\n Đã lưu: {SYNTHETIC_RESULT_DIR}/benchmark_results.csv')
    print(f' Đã lưu: {SYNTHETIC_RESULT_DIR}/benchmark_summary.csv')

Tổng tổ hợp mỗi dataset: GCBD=506, GK-GCBD=2024, CLIQUE=276

[3-spiral]
  GCBD gốc (506 combos)... best AMI=0.9635
  GK-GCBD σ=0.25 (506 combos)... best AMI=1.0000
  GK-GCBD σ=0.5 (506 combos)... 

KeyboardInterrupt: 

---
#  PHẦN F — BENCHMARK 2: DỮ LIỆU OSM (THỰC TẾ)

**Mục tiêu:** Đánh giá chất lượng phân cụm trên dữ liệu POI thực tế từ OpenStreetMap  
**Chỉ số đánh giá:** Silhouette, BSS, WSS, WB, DB, Dunn, CH  
*(Không có ground truth → dùng internal clustering metrics)*


In [31]:
# ============================================================
# F1. HÀM TÍNH CÁC CHỈ SỐ ĐÁNH GIÁ NỘI BỘ (INTERNAL METRICS)
# ============================================================

import warnings
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import pairwise_distances

def compute_bss_wss(X, labels):
    """
    Tính BSS (Between-Sum-of-Squares) và WSS (Within-Sum-of-Squares).

    Input:
        X      (np.ndarray): dữ liệu, shape (n, m)
        labels (np.ndarray): nhãn cluster (>0)

    Output:
        bss (float): tổng bình phương giữa các cụm (lớn hơn = tốt hơn)
        wss (float): tổng bình phương trong cụm (nhỏ hơn = tốt hơn)
    """
    grand_mean = X.mean(axis=0)
    bss = wss = 0.0
    for lbl in np.unique(labels[labels > 0]):
        mask         = labels == lbl
        cluster_pts  = X[mask]
        cluster_mean = cluster_pts.mean(axis=0)
        bss += mask.sum() * np.sum((cluster_mean - grand_mean) ** 2)
        wss += np.sum((cluster_pts - cluster_mean) ** 2)
    return bss, wss


def dunn_index(X, labels):
    """
    Tính Dunn Index.

    Input:
        X      (np.ndarray): dữ liệu, shape (n, m)
        labels (np.ndarray): nhãn cluster

    Output:
        dunn (float): Dunn Index (lớn hơn = cụm compact và tách biệt hơn)
    """
    unique_lbl = np.unique(labels[labels > 0])
    if len(unique_lbl) < 2:
        return 0.0
    clusters  = [X[labels == l] for l in unique_lbl]
    max_intra = [pairwise_distances(c).max() if len(c) >= 2 else 0.0 for c in clusters]
    min_inter = min(
        pairwise_distances(clusters[i], clusters[j]).min()
        for i in range(len(clusters)) for j in range(i+1, len(clusters))
    )
    max_diam = max(max_intra) or 1.0
    return min_inter / max_diam


def compute_all_metrics(X, pred):
    """
    Tính toàn bộ chỉ số đánh giá nội bộ từ dữ liệu X và nhãn pred.

    Input:
        X    (np.ndarray): dữ liệu gốc, shape (n, m)
        pred (np.ndarray): nhãn cluster, shape (n,)

    Output:
        dict chứa: n_clusters, n_assigned, Silhouette, BSS, WSS, WB, DB, Dunn, CH
        None nếu không đủ điểm hoặc không đủ cụm
    """
    active = pred > 0
    if active.sum() < 2:
        return None
    X_a, lbl_a = X[active], pred[active]
    if len(np.unique(lbl_a)) < 2:
        return None

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sil = silhouette_score(X_a, lbl_a)
        db  = davies_bouldin_score(X_a, lbl_a)
        ch  = calinski_harabasz_score(X_a, lbl_a)

    bss, wss = compute_bss_wss(X_a, lbl_a)
    n_k = len(np.unique(lbl_a))
    wb  = (n_k * wss / bss) if bss else np.nan
    dun = dunn_index(X_a, lbl_a)

    return {
        'n_clusters': n_k,
        'n_assigned': int(active.sum()),
        'Silhouette': round(sil, 6),
        'BSS': round(bss, 6), 'WSS': round(wss, 6),
        'WB' : round(wb,  6), 'DB' : round(db,  6),
        'Dunn': round(dun, 6), 'CH': round(ch, 6),
    }

print('Hàm tính chỉ số internal metrics đã sẵn sàng')

Hàm tính chỉ số internal metrics đã sẵn sàng


In [32]:
# ============================================================
# F2. CHẠY BENCHMARK OSM — GRID SEARCH & GHI KẾT QUẢ EXCEL
# ============================================================

import time
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OSM_RESULT_DIR = 'results/OSM'
os.makedirs(OSM_RESULT_DIR, exist_ok=True)


def clique_predict_osm(data, intervals, threshold):
    """
    Chạy CLIQUE cho dữ liệu OSM, trả về nhãn 1-indexed (0 = noise).

    Input:
        data      (np.ndarray): dữ liệu, shape (n, 2)
        intervals (int)       : số khoảng chia
        threshold (int)       : ngưỡng mật độ

    Output:
        labels (np.ndarray): nhãn cluster (>0) hoặc noise (=0)
    """
    if not CLIQUE_AVAILABLE:
        return None
    scaler      = MinMaxScaler()
    data_scaled = scaler.fit_transform(data).tolist()
    inst        = clique(data_scaled, intervals, threshold)
    inst.process()
    clusters = inst.get_clusters()
    labels   = np.zeros(len(data), dtype=int)
    for cid, indices in enumerate(clusters, start=1):
        for idx in indices:
            labels[idx] = cid
    return labels


txt_files_list = sorted(Path('Datasets/OSM_dataset').glob('*.txt'))
if not txt_files_list:
    print('⚠️  Không tìm thấy file .txt — hãy upload dữ liệu OSM ở Phần B')
else:
    osm_results = []

    for fpath in txt_files_list:
        dataset_name, X = load_osm_dataset(str(fpath))
        print(f'\n{'='*55}\n▶ {dataset_name} (n={len(X)})')

        best = {'original': None, 'gaussian': None, 'clique': None}

        # ── GCBD gốc ──────────────────────────────────────────────────────────
        print(f'  Chạy GCBD gốc...')
        for combo in itertools.product(*GRID_PARAMS.values()):
            params = dict(zip(GRID_PARAMS.keys(), combo))
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    t0   = time.perf_counter()
                    pred = fastgrid(X, **params)
                    elapsed = time.perf_counter() - t0
            except Exception:
                continue
            m = compute_all_metrics(X, pred)
            if m and (best['original'] is None or m['Silhouette'] > best['original']['Silhouette']):
                best['original'] = {**m, 'time': elapsed, **params}

        # ── GK-GCBD ───────────────────────────────────────────────────────────
        print(f'  Chạy GK-GCBD...')
        for sigma in SIGMA_VALUES:
            for combo in itertools.product(*GRID_PARAMS.values()):
                params = dict(zip(GRID_PARAMS.keys(), combo))
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        t0   = time.perf_counter()
                        pred = fastgrid_gaussian(X, **params, sigma=sigma)
                        elapsed = time.perf_counter() - t0
                except Exception:
                    continue
                m = compute_all_metrics(X, pred)
                if m and (best['gaussian'] is None or m['Silhouette'] > best['gaussian']['Silhouette']):
                    best['gaussian'] = {**m, 'time': elapsed, **params, 'sigma': sigma}

        # ── CLIQUE ────────────────────────────────────────────────────────────
        if CLIQUE_AVAILABLE:
            print(f'  Chạy CLIQUE...')
            for intervals in CLIQUE_PARAMS['intervals']:
                for threshold in CLIQUE_PARAMS['threshold']:
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore')
                            t0   = time.perf_counter()
                            pred = clique_predict_osm(X, intervals, threshold)
                            elapsed = time.perf_counter() - t0
                        if pred is None: continue
                    except Exception:
                        continue
                    m = compute_all_metrics(X, pred)
                    if m and (best['clique'] is None or m['Silhouette'] > best['clique']['Silhouette']):
                        best['clique'] = {**m, 'time': elapsed,
                                          'intervals': intervals, 'threshold': threshold}

        for key in best:
            r = best[key]
            if r:
                print(f'  📊 {key:10s}: Sil={r["Silhouette"]:.4f}, k={r["n_clusters"]}')

        osm_results.append((dataset_name, best['original'], best['gaussian'], best['clique']))

    print(f'\nHoàn thành Benchmark OSM cho {len(osm_results)} dataset(s)')
    print('   → Chạy cell tiếp theo để xuất file Excel')


▶ Acute1_4_CanThoInput (n=2563)
  Chạy GCBD gốc...


KeyboardInterrupt: 

In [ ]:
# ============================================================
# F3. XUẤT KẾT QUẢ OSM RA FILE EXCEL
# ============================================================

def write_osm_xlsx(all_results, out_path):
    """
    Ghi kết quả benchmark OSM ra file Excel có định dạng.

    Input:
        all_results (list): list of (dataset_name, best_orig, best_gauss, best_clique)
        out_path    (str) : đường dẫn file .xlsx
    """
    wb = Workbook()
    ws = wb.active
    ws.title = 'Sheet1'

    headers = ['Dataset', 'Algorithm', 'Số cụm', 'Số phần tử',
               'l (no_grid)', 'T (max_iters)', 'Sigma',
               'Intervals', 'Threshold',
               'Time (s)', 'Silhouette', 'BSS', 'WSS', 'WB', 'DB', 'Dunn', 'CH',
               '%Time vs Gốc']

    thin   = Side(border_style='thin', color='999999')
    brd    = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal='center', vertical='center')
    hfill  = PatternFill('solid', start_color='4472C4')
    hfont  = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    dfont  = Font(name='Arial', size=10)

    for ci, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=ci, value=h)
        cell.font = hfont; cell.fill = hfill
        cell.alignment = center; cell.border = brd

    fills = {
        'Cải tiến': PatternFill('solid', start_color='DCE6F1'),
        'Gốc'     : PatternFill('solid', start_color='FFFFFF'),
        'CLIQUE'  : PatternFill('solid', start_color='EBF1DE'),
    }
    FLOAT_COLS = set(range(10, 19))
    row_idx = 2

    for entry in all_results:
        ds_name, best_orig, best_gauss, best_clique = entry
        orig_time = best_orig.get('time') if best_orig else None

        for ai, (label, d) in enumerate([('Cải tiến', best_gauss),
                                          ('Gốc',      best_orig),
                                          ('CLIQUE',   best_clique)]):
            if d is None:
                row_data = [ds_name if ai==0 else None, label] + [None]*16
            else:
                t = d.get('time')
                pct = (t / orig_time) if (label != 'Gốc' and orig_time and t) else None
                l_display = (d.get('no_grid', 0) - 1) if label != 'CLIQUE' else None
                row_data = [
                    ds_name if ai==0 else None, label,
                    d['n_clusters'], d['n_assigned'],
                    l_display, d.get('max_iters'), d.get('sigma') if label=='Cải tiến' else None,
                    d.get('intervals'), d.get('threshold'),
                    round(t, 6) if t else None,
                    d['Silhouette'], d['BSS'], d['WSS'], d['WB'], d['DB'], d['Dunn'], d['CH'],
                    round(pct, 6) if pct else None,
                ]
            fill = fills[label]
            for ci, v in enumerate(row_data, 1):
                cell = ws.cell(row=row_idx, column=ci, value=v)
                cell.font = dfont; cell.fill = fill
                cell.alignment = center; cell.border = brd
                if ci in FLOAT_COLS and v is not None:
                    cell.number_format = '0.000000'
            row_idx += 1

    for i, w in enumerate([14,12,10,14,12,12,10,12,12,12,12,16,16,10,10,10,10,14], 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.row_dimensions[1].height = 22
    wb.save(out_path)
    print(f'Đã lưu Excel: {out_path}')


if 'osm_results' in dir() and osm_results:
    out_path = f'{OSM_RESULT_DIR}/results.xlsx'
    write_osm_xlsx(osm_results, out_path)
    from google.colab import files
    files.download(out_path)
else:
    print(' Chưa có kết quả OSM — hãy chạy cell F2 trước')

---
#  PHẦN G — BENCHMARK 3: SO SÁNH THỜI GIAN CHẠY (RUNTIME)

**Mục tiêu:** Đo thời gian thực thi trung bình của 3 thuật toán trên cùng dữ liệu OSM  
**Phương pháp:** Chạy N lần, lấy giá trị trung bình để loại bỏ nhiễu


In [37]:
# ============================================================
# G1. ĐO THỜI GIAN CHẠY TRUNG BÌNH
# ============================================================
#
# Tham số benchmark:
#   N_RUNS      : số lần chạy mỗi thuật toán để lấy trung bình
#   Tham số thuật toán: sử dụng bộ tham số cố định hợp lý
#
# Input (mỗi lần chạy):
#   - X (np.ndarray): dữ liệu OSM, shape (n, 2)
#
# Output:
#   - mean_time (float): thời gian trung bình (giây)
#   - std_time  (float): độ lệch chuẩn (giây)

import time

# ── Tham số benchmark runtime ─────────────────────────────────────────────────
N_RUNS           = 10      # Tăng lên 50 để kết quả chính xác hơn (tốn thời gian)
RT_NO_GRID       = 21
RT_PERCENTILE    = 0.1
RT_MAX_ITERS     = 5
RT_SIGMA         = 0.5
RT_CLIQUE_INTER  = 20
RT_CLIQUE_THRES  = 2


def measure_runtime(fn, X, n_runs):
    """
    Đo thời gian chạy trung bình của hàm fn trên dữ liệu X.

    Input:
        fn     (callable)  : hàm thuật toán cần đo
        X      (np.ndarray): dữ liệu đầu vào
        n_runs (int)       : số lần chạy

    Output:
        mean_sec (float): thời gian trung bình (giây)
        std_sec  (float): độ lệch chuẩn (giây)
    """
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fn(X)
        times.append(time.perf_counter() - t0)
    arr = np.array(times)
    return float(arr.mean()), float(arr.std())


RUNTIME_DIR = 'results/runtime'
os.makedirs(RUNTIME_DIR, exist_ok=True)

runtime_rows = []

txt_files_list = sorted(Path('Datasets/OSM_dataset').glob('*.txt'))
if not txt_files_list:
    print(' Chưa có dữ liệu OSM — hãy upload file .txt ở Phần B')
else:
    print(f' Mỗi thuật toán sẽ chạy {N_RUNS} lần / dataset')
    print(f'   Tham số: no_grid={RT_NO_GRID}, max_iters={RT_MAX_ITERS}, sigma={RT_SIGMA}')
    print('='*55)

    for fpath in txt_files_list:
        dataset_name, X = load_osm_dataset(str(fpath))
        print(f'\n▶ {dataset_name} (n={len(X)})')

        # GCBD gốc
        mean_g, std_g = measure_runtime(
            lambda x: fastgrid(x, no_grid=RT_NO_GRID,
                               percentile=RT_PERCENTILE, max_iters=RT_MAX_ITERS), X, N_RUNS)
        print(f'   GCBD gốc    : {mean_g:.3f}s ± {std_g:.3f}s')

        # GK-GCBD
        mean_gk, std_gk = measure_runtime(
            lambda x: fastgrid_gaussian(x, no_grid=RT_NO_GRID,
                                        percentile=RT_PERCENTILE, max_iters=RT_MAX_ITERS,
                                        sigma=RT_SIGMA), X, N_RUNS)
        print(f'   GK-GCBD     : {mean_gk:.3f}s ± {std_gk:.3f}s')

        # CLIQUE
        if CLIQUE_AVAILABLE:
            mean_cl, std_cl = measure_runtime(
                lambda x: clique_predict_osm(x, RT_CLIQUE_INTER, RT_CLIQUE_THRES), X, N_RUNS)
            print(f'   CLIQUE      : {mean_cl:.3f}s ± {std_cl:.3f}s')
        else:
            mean_cl = std_cl = None
            print(f'   CLIQUE      : bỏ qua')

        runtime_rows.append({
            'dataset'    : dataset_name,
            'gcbd_mean'  : mean_g,  'gcbd_std'  : std_g,
            'gauss_mean' : mean_gk, 'gauss_std' : std_gk,
            'clique_mean': mean_cl, 'clique_std': std_cl,
        })

    print(f'\n Hoàn thành đo runtime cho {len(runtime_rows)} dataset(s)')

 Mỗi thuật toán sẽ chạy 10 lần / dataset
   Tham số: no_grid=21, max_iters=5, sigma=0.5

▶ Acute1_4_CanThoInput (n=2563)
   GCBD gốc    : 0.013s ± 0.001s
   GK-GCBD     : 0.014s ± 0.003s
   CLIQUE      : 0.056s ± 0.069s

▶ Acute1_4_DongNaiInput (n=2704)
   GCBD gốc    : 0.021s ± 0.003s
   GK-GCBD     : 0.018s ± 0.001s
   CLIQUE      : 0.065s ± 0.087s

▶ Acute1_4_Nhom1Input (n=4446)
   GCBD gốc    : 0.044s ± 0.005s
   GK-GCBD     : 0.042s ± 0.001s
   CLIQUE      : 0.080s ± 0.067s

▶ Acute1_4_Nhom2Input (n=4316)
   GCBD gốc    : 0.035s ± 0.001s
   GK-GCBD     : 0.035s ± 0.001s
   CLIQUE      : 0.085s ± 0.067s

▶ Acute1_4_Nhom3Input (n=5649)
   GCBD gốc    : 0.068s ± 0.003s
   GK-GCBD     : 0.056s ± 0.011s
   CLIQUE      : 0.136s ± 0.114s

 Hoàn thành đo runtime cho 5 dataset(s)


In [34]:
# ============================================================
# G2. VẼ BIỂU ĐỒ & XUẤT EXCEL KẾT QUẢ RUNTIME
# ============================================================

import matplotlib.ticker as ticker
from openpyxl import Workbook

def write_runtime_xlsx(rows, out_path):
    """Ghi bảng thời gian chạy ra file Excel."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Runtime'

    headers = ['Dataset', 'GCBD (s)', 'GCBD Std',
               'GK-GCBD (s)', 'GK-GCBD Std', 'CLIQUE (s)', 'CLIQUE Std']
    hfill = PatternFill('solid', start_color='4472C4')
    hfont = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    thin  = Side(border_style='thin', color='999999')
    brd   = Border(left=thin, right=thin, top=thin, bottom=thin)
    ctr   = Alignment(horizontal='center', vertical='center')

    for ci, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=ci, value=h)
        cell.font = hfont; cell.fill = hfill
        cell.alignment = ctr; cell.border = brd

    alt = [PatternFill('solid', start_color='DCE6F1'),
           PatternFill('solid', start_color='FFFFFF')]

    for ri, row in enumerate(rows, 2):
        fill = alt[ri % 2]
        vals = [row['dataset'], row['gcbd_mean'], row['gcbd_std'],
                row['gauss_mean'], row['gauss_std'],
                row['clique_mean'], row['clique_std']]
        for ci, v in enumerate(vals, 1):
            cell = ws.cell(row=ri, column=ci,
                           value=round(v, 6) if isinstance(v, float) else v)
            cell.font = Font(name='Arial', size=10); cell.fill = fill
            cell.alignment = ctr; cell.border = brd
            if ci > 1 and v is not None:
                cell.number_format = '0.000000'

    for ci, w in enumerate([16, 14, 12, 14, 12, 12, 12], 1):
        ws.column_dimensions[get_column_letter(ci)].width = w
    wb.save(out_path)
    print(f' Excel: {out_path}')


if 'runtime_rows' in dir() and runtime_rows:
    xlsx_path  = f'{RUNTIME_DIR}/runtime_results.xlsx'
    chart_path = f'{RUNTIME_DIR}/runtime_chart.png'

    write_runtime_xlsx(runtime_rows, xlsx_path)

    # ── Vẽ biểu đồ đường so sánh ──────────────────────────────────────────────
    datasets    = [r['dataset']    for r in runtime_rows]
    gcbd_times  = [r['gcbd_mean']  for r in runtime_rows]
    gauss_times = [r['gauss_mean'] for r in runtime_rows]
    clique_times = [r['clique_mean'] for r in runtime_rows]

    x = range(len(datasets))
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#F5F5F5')
    ax.set_facecolor('#F5F5F5')

    ax.plot(x, gcbd_times,  color='#2196F3', marker='o', linewidth=2,
            markersize=6, label='GCBD')
    ax.plot(x, gauss_times, color='#F44336', marker='o', linewidth=2,
            markersize=6, label='GK-GCBD')
    if CLIQUE_AVAILABLE and any(v is not None for v in clique_times):
        ax.plot(x, clique_times, color='#4CAF50', marker='s', linewidth=2,
                markersize=6, label='CLIQUE')

    ax.set_title('Comparison chart about implementation times', fontsize=12,
                 fontweight='bold', pad=10)
    ax.set_xlabel('Datasets', fontsize=12)
    ax.set_ylabel('Runtime (Seconds)', fontsize=12)
    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=0, fontsize=11)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linestyle='-',  linewidth=0.5, color='#CCCCCC')
    ax.grid(True, which='minor', linestyle='--', linewidth=0.3, color='#DDDDDD')
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Biểu đồ: {chart_path}')

    # Tải về máy
    from google.colab import files
    files.download(xlsx_path)
    files.download(chart_path)
else:
    print('Chưa có kết quả runtime — hãy chạy cell G1 trước')

Excel: results/runtime/runtime_results.xlsx
Biểu đồ: results/runtime/runtime_chart.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
#  PHẦN H — TỔNG KẾT & TẢI KẾT QUẢ VỀ MÁY


In [38]:
# ============================================================
# H. TẢI TOÀN BỘ KẾT QUẢ VỀ MÁY (nén thành ZIP)
# ============================================================
# Lưu ý: folder results/ chỉ có nội dung sau khi đã chạy ít nhất
# một trong các benchmark (E, F, G). Nếu chưa chạy benchmark nào
# thì ZIP sẽ rỗng — không bị lỗi nhưng không có gì để tải.

import os
import shutil
from pathlib import Path
from google.colab import files

# Đảm bảo folder results tồn tại trước khi nén
for d in ['results/Synthetic', 'results/OSM', 'results/runtime']:
    os.makedirs(d, exist_ok=True)

# Kiểm tra có file kết quả nào chưa
result_files = list(Path('results').rglob('*'))
result_files = [f for f in result_files if f.is_file()]

if not result_files:
    print('Chưa có kết quả nào — hãy chạy ít nhất một benchmark (E, F hoặc G) trước!')
else:
    zip_path = '/content/GCBD_results'
    shutil.make_archive(zip_path, 'zip', '/content/GCBD/results')

    print('Nội dung file ZIP:')
    for p in result_files:
        size_kb = p.stat().st_size / 1024
        print(f'   {p}  ({size_kb:.1f} KB)')

    files.download(zip_path + '.zip')
    print('\nĐã tải file GCBD_results.zip!')


Nội dung file ZIP:
   results/raw_data_visualization.png  (92.3 KB)
   results/runtime/runtime_results.xlsx  (5.4 KB)
   results/runtime/runtime_chart.png  (99.9 KB)
   results/OSM/results.xlsx  (7.7 KB)
   results/Synthetic/clustering_elliptical_10_2.png  (169.7 KB)
   results/Synthetic/clustering_smile3.png  (179.0 KB)
   results/Synthetic/clustering_blobs.png  (150.4 KB)
   results/Synthetic/clustering_compound.png  (165.8 KB)
   results/Synthetic/benchmark_gaussian_summary.csv  (5.4 KB)
   results/Synthetic/clustering_R15.png  (161.6 KB)
   results/Synthetic/clustering_aggregation.png  (348.8 KB)
   results/Synthetic/clustering_square4.png  (396.0 KB)
   results/Synthetic/clustering_3-spiral.png  (148.4 KB)
   results/Synthetic/clustering_insect.png  (19.4 KB)
   results/Synthetic/clustering_tetra.png  (199.0 KB)
   results/Synthetic/clustering_pathbased.png  (155.3 KB)
   results/Synthetic/clustering_rings.png  (296.3 KB)
   results/Synthetic/clustering_flame.png  (129.3 KB)
   re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Đã tải file GCBD_results.zip!
